# Tính KPI dựa trên bộ dữ liệu clean

## Thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Đọc file dữ liệu

In [2]:
df = pd.read_csv("../../data/processed/Online_Retail_clean.csv", encoding="ISO-8859-1", parse_dates=["InvoiceDate"])

## LOẠI 1: Doanh thu

In [3]:
total_revenue = df["Revenue"].sum()
print(f'Total Revenue: {total_revenue}')

Total Revenue: 8887208.889999999


In [4]:
revenue_over_month = df.groupby("YearMonth")["Revenue"].sum()
print(f'Revenue Over Month:\n{revenue_over_month}')

Revenue Over Month:
YearMonth
2010-12     570422.73
2011-01     568101.31
2011-02     446084.92
2011-03     594081.76
2011-04     468374.33
2011-05     677355.15
2011-06     660046.05
2011-07     598962.90
2011-08     644051.04
2011-09     950690.20
2011-10    1035642.45
2011-11    1156205.61
2011-12     517190.44
Name: Revenue, dtype: float64


In [5]:
aov = df.groupby("InvoiceNo")["Revenue"].sum().mean()
print(f'Average Order Value: {aov}')

Average Order Value: 479.5601602633283


In [6]:
revenue_per_customer = df.groupby("CustomerID")["Revenue"].sum().mean()
print(f'Average Revenue per Customer: {revenue_per_customer}')

Average Revenue per Customer: 2048.6880797602585


## LOẠI 1: Doanh thu theo giờ

In [7]:
revenue_by_hour = df.groupby("Hour")["Revenue"].sum().reset_index()
print(f'Revenue Over Hour:\n{revenue_by_hour}')

revenue_by_hour.to_csv("../../data/kpi/revenue_by_hour.csv", index=False)

Revenue Over Hour:
    Hour     Revenue
0      6        4.25
1      7    31059.21
2      8   281997.79
3      9   842392.34
4     10  1259267.59
5     11  1101177.60
6     12  1373695.39
7     13  1168724.20
8     14   991992.82
9     15   963559.68
10    16   467380.56
11    17   233811.59
12    18   104744.99
13    19    48568.40
14    20    18832.48


In [8]:
revenue_by_weekday = df.groupby("Weekday")["Revenue"].sum()
print(f'Revenue By Weekday:\n{revenue_by_weekday}')

Revenue By Weekday:
Weekday
Friday       1483080.81
Monday       1363604.40
Sunday        785490.32
Thursday     1973015.73
Tuesday      1697733.80
Wednesday    1584283.83
Name: Revenue, dtype: float64


## Loại 2: Tính Doanh thu theo tháng

In [9]:
revenue_by_month = df.groupby("YearMonth")["Revenue"].sum().reset_index()
print(f'Revenue By Month:\n{revenue_by_month}')
revenue_by_month.to_csv("../../data/kpi/revenue_by_month.csv", index=False)

Revenue By Month:
   YearMonth     Revenue
0    2010-12   570422.73
1    2011-01   568101.31
2    2011-02   446084.92
3    2011-03   594081.76
4    2011-04   468374.33
5    2011-05   677355.15
6    2011-06   660046.05
7    2011-07   598962.90
8    2011-08   644051.04
9    2011-09   950690.20
10   2011-10  1035642.45
11   2011-11  1156205.61
12   2011-12   517190.44


### Loại bỏ tháng 12 năm 2010

In [10]:
df_filtered = df[~(
    (df["InvoiceDate"].dt.year == 2010) & 
    (df["InvoiceDate"].dt.month == 12)
)]

monthly_revenue_excl_dec2010 = df_filtered.groupby("YearMonth")["Revenue"].sum().reset_index()

print(f"Revenue By Month:\n{monthly_revenue_excl_dec2010}")

monthly_revenue_excl_dec2010.to_csv("../../data/kpi/monthly_revenue_excl_dec2010.csv", index=False)

Revenue By Month:
   YearMonth     Revenue
0    2011-01   568101.31
1    2011-02   446084.92
2    2011-03   594081.76
3    2011-04   468374.33
4    2011-05   677355.15
5    2011-06   660046.05
6    2011-07   598962.90
7    2011-08   644051.04
8    2011-09   950690.20
9    2011-10  1035642.45
10   2011-11  1156205.61
11   2011-12   517190.44


## LOẠI 3: Thời gian đến mua hàng

In [11]:
first_purchase = df.groupby("CustomerID")["InvoiceDate"].min()
last_purchase = df.groupby("CustomerID")["InvoiceDate"].max()
customer_lifetime = (last_purchase - first_purchase).dt.days
print(f'first_purchase:\n{first_purchase}')
print(f'last_purchase:\n{last_purchase}')
print(f'Average Customer Lifetime (days): {customer_lifetime.mean()}')


first_purchase:
CustomerID
12346.0   2011-01-18 10:01:00
12347.0   2010-12-07 14:57:00
12348.0   2010-12-16 19:09:00
12349.0   2011-11-21 09:51:00
12350.0   2011-02-02 16:01:00
                  ...        
18280.0   2011-03-07 09:52:00
18281.0   2011-06-12 10:53:00
18282.0   2011-08-05 13:35:00
18283.0   2011-01-06 14:14:00
18287.0   2011-05-22 10:39:00
Name: InvoiceDate, Length: 4338, dtype: datetime64[ns]
last_purchase:
CustomerID
12346.0   2011-01-18 10:01:00
12347.0   2011-12-07 15:52:00
12348.0   2011-09-25 13:13:00
12349.0   2011-11-21 09:51:00
12350.0   2011-02-02 16:01:00
                  ...        
18280.0   2011-03-07 09:52:00
18281.0   2011-06-12 10:53:00
18282.0   2011-12-02 11:43:00
18283.0   2011-12-06 12:02:00
18287.0   2011-10-28 09:29:00
Name: InvoiceDate, Length: 4338, dtype: datetime64[ns]
Average Customer Lifetime (days): 130.4485938220378


In [12]:
df = df.sort_values(["CustomerID", "InvoiceDate"])

df["prev_purchase"] = df.groupby("CustomerID")["InvoiceDate"].shift(1)

df["days_between"] = (df["InvoiceDate"] - df["prev_purchase"]).dt.days


In [13]:
purchase_freq = df.groupby("CustomerID")["InvoiceNo"].nunique().mean()
print(f'Average Purchase Frequency: {purchase_freq}')

Average Purchase Frequency: 4.272014753342554


## Loại 4: KPI tổng 

In [14]:
kpi_summary = pd.DataFrame({
    "metric": [
        "Total Revenue",
        "AOV",
        "Purchase Frequency"
    ],
    "value": [
        df["Revenue"].sum(),
        df.groupby("InvoiceNo")["Revenue"].sum().mean(),
        df.groupby("CustomerID")["InvoiceNo"].nunique().mean()
    ]
})

kpi_summary.to_csv("../../data/kpi/kpi_summary.csv", index=False)

## Loại 5: KPI theo mã khách hàng

In [15]:
customer_metrics = df.groupby("CustomerID").agg({
    "InvoiceNo": "nunique",
    "Revenue": "sum"
}).reset_index()

customer_metrics.columns = ["CustomerID", "NumOrders", "TotalRevenue"]

customer_metrics.to_csv("../../data/kpi/customer_metrics.csv", index=False)

## Loại 6: KPI thời gian giữa các lần thanh toán

In [16]:
df = df.sort_values(["CustomerID", "InvoiceDate"])

df["prev_purchase"] = df.groupby("CustomerID")["InvoiceDate"].shift(1)

df["days_between"] = (df["InvoiceDate"] - df["prev_purchase"]).dt.days

time_between = df[["CustomerID", "days_between"]].dropna()

time_between.to_csv("../../data/kpi/time_between_purchases.csv", index=False)